### HW6 Q1 — 4-bit VGG16: Hardware Mapping with Tiling

In [5]:
import os, sys, time, math, shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

sys.path.insert(0, os.path.abspath(".."))
from models import *
from models.quant_layer import act_quantization

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

batch_size = 128
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True,
    transform=transforms.Compose([transforms.RandomCrop(32, padding=4),
                                  transforms.RandomHorizontalFlip(),
                                  transforms.ToTensor(), normalize]))
test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True,
    transform=transforms.Compose([transforms.ToTensor(), normalize]))
trainloader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=2)
testloader  = torch.utils.data.DataLoader(test_set,  batch_size=batch_size, shuffle=False, num_workers=2)

device: mps


In [6]:
def run_epoch(loader, model, criterion, optimizer=None):
    """One pass over loader. Trains if an optimizer is given, otherwise evaluates. Returns (loss, acc%)."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, seen = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            seen += y.size(0)
    return total_loss / seen, 100.0 * correct / seen


# paths and model (shared by the training cell and the test cell)
model_name = "vgg16_quant"
fdir = os.path.join("result", model_name)
os.makedirs(fdir, exist_ok=True)
best_path = os.path.join(fdir, "model_best.pth.tar")

model = VGG16_quant().to(device)
criterion = nn.CrossEntropyLoss()


In [ ]:
RETRAIN = False

if RETRAIN or not os.path.exists(best_path):
    lr, epochs, warmup_iters = 4e-2, 100, 500

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    total_iters = epochs * len(trainloader)
    def lr_lambda(it):
        if it < warmup_iters:
            return (it + 1) / warmup_iters
        p = (it - warmup_iters) / max(1, total_iters - warmup_iters)
        return 0.5 * (1 + math.cos(math.pi * p))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)  # stepped per iteration

    # wrap optimizer.step so the per-iteration scheduler advances inside run_epoch
    _step = optimizer.step
    def step_and_schedule(*a, **k):
        _step(*a, **k); scheduler.step()
    optimizer.step = step_and_schedule

    best_acc = 0.0
    for epoch in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(trainloader, model, criterion, optimizer)
        te_loss, te_acc = run_epoch(testloader, model, criterion)
        state = {"epoch": epoch + 1, "state_dict": model.state_dict(), "best_prec": max(best_acc, te_acc)}
        torch.save(state, os.path.join(fdir, "checkpoint.pth"))
        if te_acc > best_acc:
            best_acc = te_acc
            shutil.copyfile(os.path.join(fdir, "checkpoint.pth"), best_path)
        print(f"epoch {epoch+1:3d}/{epochs}  lr {optimizer.param_groups[0]['lr']:.4f}  "
              f"train {tr_acc:5.2f}%  test {te_acc:5.2f}%  best {best_acc:5.2f}%  ({time.time()-t0:.0f}s)")
else:
    print("4-bit checkpoint found, skipping training:", best_path)

### Test accuracy of the saved 4-bit model

In [7]:
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["state_dict"])
model.to(device).eval()
_, test_acc = run_epoch(testloader, model, criterion)
print(f"VGG16_quant (4-bit QAT) test accuracy: {test_acc:.2f}%")

VGG16_quant (4-bit QAT) test accuracy: 91.24%


### Target layer: `model.features[3]`

In [8]:
print(model.features[:6])
layer = model.features[3]
print("\ntarget layer:", layer)
assert layer.in_channels == 64 and layer.out_channels == 64

Sequential(
  (0): QuantConv2d(
    3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (weight_quant): weight_quantize_fn()
  )
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): QuantConv2d(
    64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (weight_quant): weight_quantize_fn()
  )
  (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (5): ReLU(inplace=True)
)

target layer: QuantConv2d(
  64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
  (weight_quant): weight_quantize_fn()
)


In [9]:
captured = {}
def grab(module, inp, out):
    captured["x"] = inp[0].detach().clone()
    captured["y"] = out.detach().clone()

handle = layer.register_forward_hook(grab)
images, labels = next(iter(testloader))
with torch.no_grad():
    model(images.to(device))
handle.remove()

bits = 4
IMG = 0  # use one image

# weights
w_delta = layer.weight_quant.wgt_alpha.detach() / (2 ** (bits - 1) - 1)
w_raw = layer.weight_q.detach() / w_delta
w_int = w_raw.round().cpu()                                # [64, 64, 3, 3] = [C_out, C_in, ky, kx]

# activations
x_alpha = layer.act_alpha.detach()
x_delta = x_alpha / (2 ** bits - 1)
x_q = act_quantization(bits)(captured["x"], x_alpha)
x_raw = x_q / x_delta
x_int = x_raw[IMG].round().cpu()                           # [64, 32, 32] = [C_in, H, W]

print("w_int", tuple(w_int.shape), "range", int(w_int.min()), "..", int(w_int.max()),
      "| max rounding fix", (w_raw - w_raw.round()).abs().max().item())
print("x_int", tuple(x_int.shape), "range", int(x_int.min()), "..", int(x_int.max()),
      "| max rounding fix", (x_raw - x_raw.round()).abs().max().item())

w_int (64, 64, 3, 3) range -7 .. 7 | max rounding fix 4.76837158203125e-07
x_int (64, 32, 32) range 0 .. 14 | max rounding fix 9.5367431640625e-07


In [10]:
ARR = 16                       # array is ARR x ARR
K = w_int.shape[-1]            # 3
PAD = 1
C_in, H, W = x_int.shape
C_out = w_int.shape[0]
n_ic, n_oc = C_in // ARR, C_out // ARR      # 4, 4
Hp, Wp = H + 2 * PAD, W + 2 * PAD           # 34, 34
Ho, Wo = Hp - K + 1, Wp - K + 1             # 32, 32

# --- input: pad, flatten to time steps, split ic into tiles, repeat over oc tiles
x_pad = torch.zeros(C_in, Hp, Wp)
x_pad[:, PAD:PAD + H, PAD:PAD + W] = x_int
x_stream = x_pad.reshape(C_in, Hp * Wp)                         # [64, 1156], nij = 34*y + x
a_tile = (x_stream.reshape(n_ic, ARR, Hp * Wp)                  # [ic_tile, r, nij]
                  .unsqueeze(1).expand(n_ic, n_oc, ARR, Hp * Wp)
                  .contiguous())                                # [ic_tile, oc_tile, r, nij]

# --- weights: [C_out, C_in, ky, kx] -> [oc_tile, c, ic_tile, r, kij] -> [ic_tile, oc_tile, r, c, kij]
w_pe = (w_int.reshape(n_oc, ARR, n_ic, ARR, K * K)
             .permute(2, 0, 3, 1, 4)
             .contiguous())

# spot check the PE assignment rule: PE(r, c) of tile (ti, to) at (ky, kx) holds w_int[16*to + c, 16*ti + r, ky, kx]
ti, to, r, c, ky, kx = 2, 1, 5, 11, 1, 2
assert w_pe[ti, to, r, c, ky * K + kx] == w_int[ARR * to + c, ARR * ti + r, ky, kx]
assert torch.equal(a_tile[ti, 3, r], x_pad[ARR * ti + r].flatten())
print("a_tile:", tuple(a_tile.shape))
print("w_pe  :", tuple(w_pe.shape))

a_tile: (4, 4, 16, 1156)
w_pe  : (4, 4, 16, 16, 9)


In [11]:
psum = torch.einsum("abrck,abrn->abcnk", w_pe, a_tile)          # [4, 4, 16, 1156, 9]
print("psum:", tuple(psum.shape))

# cross-check: simulate one pass time step by time step, like the array would
ti, to, kij = 3, 0, 4
block = w_pe[ti, to, :, :, kij]                                 # PE(r, c) weights, [16, 16]
col_out = torch.empty(ARR, Hp * Wp)
for nij in range(Hp * Wp):
    col_out[:, nij] = block.t() @ a_tile[ti, to, :, nij]        # 16 column sums at this time step
print("per-time-step sim == einsum for this pass:", torch.equal(col_out, psum[ti, to, :, :, kij]))
print("max |psum| =", psum.abs().max().item(), "(fits in 16-bit psum_bw)")

psum: (4, 4, 16, 1156, 9)
per-time-step sim == einsum for this pass: True
max |psum| = 155.0 (fits in 16-bit psum_bw)


In [12]:
oy, ox = torch.meshgrid(torch.arange(Ho), torch.arange(Wo), indexing="ij")
ky, kx = torch.meshgrid(torch.arange(K), torch.arange(K), indexing="ij")
oy, ox = oy.reshape(-1, 1), ox.reshape(-1, 1)        # [1024, 1]
ky, kx = ky.reshape(1, -1), kx.reshape(1, -1)        # [1, 9]   (kij = 3*ky + kx)
nij_addr = (oy + ky) * Wp + (ox + kx)                # [1024, 9] read address for each (o_nij, kij)

kij_idx = torch.arange(K * K).reshape(1, -1).expand_as(nij_addr)
picked = psum[:, :, :, nij_addr, kij_idx]            # [ic_tile, oc_tile, c, o_nij, kij]
acc = picked.sum(dim=(0, 4))                         # sum over ic tiles and kernel taps -> [oc_tile, c, o_nij]
out = acc.reshape(C_out, Ho * Wo)                    # place oc tiles: row = 16*oc_tile + c
print("out:", tuple(out.shape))

out: (64, 1024)


### Check against the integer reference convolution
`nn.Conv2d(64, 64, 3, stride=1, padding=1, bias=False)` with weight `w_int`, applied to the same image `x_int` reshaped to [64, 1024].

In [13]:
conv_ref = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False).double()
with torch.no_grad():
    conv_ref.weight = nn.Parameter(w_int.double())
    ref = conv_ref(x_int.double().unsqueeze(0))[0].reshape(64, -1)    # [64, 1024]

abs_diff = (out.double() - ref).abs()
print("a_tile shape:", tuple(a_tile.shape))
print("psum   shape:", tuple(psum.shape))
print("out    shape:", tuple(out.shape))
print("sum |out - ref| =", abs_diff.sum().item())
print("max |out - ref| =", abs_diff.max().item())

a_tile shape: (4, 4, 16, 1156)
psum   shape: (4, 4, 16, 1156, 9)
out    shape: (64, 1024)
sum |out - ref| = 0.0
max |out - ref| = 0.0


Measuring quantization rounding, and confirming that `x_int`/`w_int` reproduce the real layer: $y \approx out \cdot \delta_x \delta_w$.

In [14]:
y_float = captured["y"][IMG].cpu().reshape(64, -1)
recovered = out * (x_delta * w_delta).cpu()
print("mean |y - out*dx*dw| =", (y_float - recovered).abs().mean().item())
print("mean |y|             =", y_float.abs().mean().item())

mean |y - out*dx*dw| = 4.3914287743973546e-06
mean |y|             = 31.60175895690918
